# 01 · Data extraction & validation — CCFA monthly brand registrations

**Project:** BYD France – Business Intelligence & Market Analytics
**Phase:** 2 (data extraction and cleaning) · **Notebook 1 of 4**

### Business purpose
Answering *"How is BYD performing and growing in France, and how does it compare with competitors?"* requires a **monthly time series of registrations by brand**. CCFA publishes this only as one PDF per month. This notebook converts the 34 PDFs (January 2024 – October 2026) into one analysis-ready table and **proves the extraction is correct** before any KPI is built.

### Inputs → outputs
| Input | Output (written to `data/processed/`) |
|---|---|
| `data/raw/Immatriculations-VPN*.pdf` (34 files) | `brand_month_raw.csv` – one row per brand per month, labels exactly as reported |
| `docs/source_register.csv` | `extraction_checks.csv` – one row per file with reconciliation results |
| | `ytd_revisions_log.csv` – months where CCFA's year-to-date figures were revised |

### Data integrity rules applied here
* Figures are copied from the source, never estimated. Blank cells stay **missing (NaN), never 0**.
* Brand names are kept **as reported** (`ABARTH/FIAT`, `M.G.`…). Standardisation happens in notebook 02, so this file stays a faithful copy of the source.
* Every row keeps its `source_id`, so any number can be traced back to a PDF and URL.

## Step 1 · Set up the environment

**What it does:** installs `pdfplumber` (a PDF-reading library) and imports the tools we need.
**Why:** Colab already has `pandas`, but not `pdfplumber`.
**How to run:** click the ▶ button on the cell (or press Shift+Enter). It takes ~10 seconds.
**What you should see:** a line printing the pdfplumber version. Any red error → stop and send it to me.

In [ ]:
!pip install -q pdfplumber

import re, glob, os
import pandas as pd
import pdfplumber

pd.set_option("display.max_columns", 20)
print("pdfplumber", pdfplumber.__version__, "| pandas", pd.__version__)

## Step 2 · Get the repository files into Colab

Your GitHub repo is **private**, so Colab cannot download it anonymously. Use **Option A** for now.

**Option A (recommended now):**
1. On GitHub open the repo → green **Code** button → **Download ZIP**.
2. Run the cell below and choose the ZIP when the upload button appears.

**Option B (later, once the repo is public):** set `USE_GIT = True`; the cell will `git clone` it instead.

**What you should see:** `PDF files found: 34` and `Register rows: 37`.

In [ ]:
USE_GIT = True    # clone the public repo; set False to upload a ZIP of your own copy instead
REPO = "https://github.com/Zaheermuhammadasad/byd-france-bi-analytics.git"

if USE_GIT:
    !git clone -q {REPO}
    BASE = "byd-france-bi-analytics"
else:
    from google.colab import files
    uploaded = files.upload()                      # choose byd-france-bi-main.zip
    zip_name = next(iter(uploaded))
    !unzip -q -o "{zip_name}"
    BASE = [d for d in glob.glob("byd-france-bi*") if os.path.isdir(d)][0]   # e.g. byd-france-bi-main

RAW = f"{BASE}/data/raw"
PROCESSED = f"{BASE}/data/processed"
os.makedirs(PROCESSED, exist_ok=True)

# The source PDFs are not redistributed in the repo (CCFA rights): download them from CCFA if missing
if not glob.glob(f"{RAW}/Immatriculations-VPN*.pdf"):
    import subprocess, sys
    subprocess.run([sys.executable, f"{BASE}/scripts/download_sources.py"], check=False)

pdfs = sorted(glob.glob(f"{RAW}/Immatriculations-VPN*.pdf"))
register = pd.read_csv(f"{BASE}/docs/source_register.csv")
print("PDF files found:", len(pdfs))
print("Register rows:", len(register))

## Step 3 · Check that every PDF is registered (and every register entry has a file)

**What it does:** compares the file names in `data/raw/` with the `local_file` column of the source register.
**Why:** a file without a register entry has no traceable source; a register entry without a file means data is missing. Either breaks the audit trail.
**How to read it:** both lists should be **empty**. If not, fix the repo before continuing.

In [ ]:
reg_files = set(register["local_file"].dropna().str.replace("data/raw/", "", regex=False))
disk_files = {os.path.basename(p) for p in pdfs}

ccfa_reg = {f for f in reg_files if f.startswith("Immatriculations-VPN")}
print("PDFs on disk but NOT in register:", sorted(disk_files - ccfa_reg))
print("Register entries with NO file   :", sorted(ccfa_reg - disk_files))

file_to_source = dict(zip(register["local_file"].str.replace("data/raw/", "", regex=False), register["source_id"]))

## Step 4 · Define the extraction functions

**Simple explanation:** a PDF has no "table" inside it, just words placed at coordinates on a page. The code rebuilds the table the way your eye does:
1. **Group words into lines** using their vertical position.
2. Find the **header line** starting with `Marques`. Its words tell us where each column sits horizontally (month, % change, year-to-date, % change).
3. For every following line, the words on the left are the **brand name**. Each number is placed in the column whose **right edge** it lines up with (CCFA right-aligns numbers).
4. Stop at the **Total** row.

**Why this design (technical):**
* CCFA prints thousands with a space (`11 897`), which pdfplumber reads as two words (`11`, `897`). Matching on column position joins them back correctly.
* Some cells are blank (e.g. % change for a brand new in France). Position-matching leaves that column empty → **NaN, not 0**, and the next number cannot slide into the wrong column.
* Some files contain **two tables** (e.g. an extra "electric cars by brand" table). The function finds all of them and keeps the **main brand table**: the one with the largest Total.

You don't need to memorise this code. Read the comments and understand the four steps above.

In [ ]:
# --- Extraction functions -------------------------------------------------

MONTHS = {"janv": 1, "fev": 2, "fév": 2, "mars": 3, "avr": 4, "mai": 5, "juin": 6,
          "juil": 7, "aout": 8, "août": 8, "sept": 9, "oct": 10, "nov": 11, "dec": 12, "déc": 12}


def parse_period(label):
    """'Septembre 2026' / 'Fev 2024' / 'Août 2026' -> '2026-09'."""
    m = re.search(r"([A-Za-zÀ-ÿ]+)\.?\s*(\d{4})", label)
    if not m:
        return None
    word, year = m.group(1).lower(), m.group(2)
    for prefix, num in MONTHS.items():
        if word.startswith(prefix):
            return f"{year}-{num:02d}"
    return None


def to_int(s):
    return int(re.sub(r"[^\d-]", "", s)) if s else None


def to_pct(s):
    return float(s.replace("%", "").replace(",", ".").replace(" ", "")) if s else None


def group_lines(words, tol=2.0):
    """Group words into text lines by vertical position."""
    lines = []
    for w in sorted(words, key=lambda w: (round(w["top"]), w["x0"])):
        if lines and abs(lines[-1][0]["top"] - w["top"]) <= tol:
            lines[-1].append(w)
        else:
            lines.append([w])
    return [sorted(l, key=lambda w: w["x0"]) for l in lines]


def header_columns(line, gap=8.0):
    """Split a 'Marques ...' header line into column groups; return label text + right edge of each."""
    groups = [[line[0]]]
    for w in line[1:]:
        if w["x0"] - groups[-1][-1]["x1"] > gap:
            groups.append([w])
        else:
            groups[-1].append(w)
    return [{"text": " ".join(w["text"] for w in g), "x0": g[0]["x0"], "x1": g[-1]["x1"]} for g in groups]


def extract_tables(pdf_path):
    """Return a list of tables (each: header labels + rows) found in one CCFA PDF."""
    tables, current = [], None
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            for line in group_lines(page.extract_words()):
                first = line[0]["text"]
                if first == "Marques":
                    cols = header_columns(line)
                    if current is None or current["closed"]:
                        current = {"cols": cols, "rows": [], "closed": False}
                        tables.append(current)
                    continue
                if current is None or current["closed"]:
                    continue
                cols = current["cols"]
                data_cols = cols[1:]                       # month, var, ytd, var (positions)
                label_limit = data_cols[0]["x0"] - 10
                label = " ".join(w["text"] for w in line if w["x1"] < label_limit)
                nums = [w for w in line if w["x1"] >= label_limit]
                if not label or not nums:
                    continue
                cells = [""] * len(data_cols)
                for w in nums:                              # assign word to nearest column right edge
                    i = min(range(len(data_cols)), key=lambda k: abs(data_cols[k]["x1"] - w["x1"]))
                    cells[i] += w["text"]
                current["rows"].append({"label": label, "cells": cells})
                if label.upper() == "TOTAL":
                    current["closed"] = True
    return tables


def extract_brand_table(pdf_path, source_id=None):
    """Pick the main brand table (largest Total) and return a tidy DataFrame + check results."""
    tables = [t for t in extract_tables(pdf_path) if t["rows"]]
    if not tables:
        raise ValueError(f"No 'Marques' table found in {pdf_path}")

    def total_of(t):
        tot = [r for r in t["rows"] if r["label"].upper() == "TOTAL"]
        return to_int(tot[0]["cells"][0]) if tot else -1

    main = max(tables, key=total_of)
    month_label = main["cols"][1]["text"]
    rows = []
    for r in main["rows"]:
        c = r["cells"] + [""] * (4 - len(r["cells"]))
        rows.append({
            "source_id": source_id, "source_file": str(pdf_path).split("/")[-1],
            "period": parse_period(month_label), "month_label": month_label,
            "brand_label": r["label"],
            "registrations_month": to_int(c[0]), "var_month_pct_reported": to_pct(c[1]),
            "registrations_ytd": to_int(c[2]), "var_ytd_pct_reported": to_pct(c[3]),
        })
    df = pd.DataFrame(rows)
    df["is_total_row"] = df["brand_label"].str.upper().eq("TOTAL")
    brands, total = df[~df.is_total_row], df[df.is_total_row]
    check = {
        "source_file": df.source_file.iloc[0], "period": df.period.iloc[0],
        "tables_found": len(tables), "brand_rows": len(brands),
        "total_month": int(total.registrations_month.iloc[0]),
        "sum_month": int(brands.registrations_month.sum()),
        "total_ytd": int(total.registrations_ytd.iloc[0]),
        "sum_ytd": int(brands.registrations_ytd.sum()),
    }
    check["month_ok"] = check["total_month"] == check["sum_month"]
    check["ytd_ok"] = check["total_ytd"] == check["sum_ytd"]
    return df, check

## Step 5 · Test on ONE file before running all 34

**Why:** professional practice. Prove the method on a file whose numbers you already know, *then* scale it up.
The October 2026 file was checked by hand in Phase 1: **BYD = 5 419** in the month, **28 086** year-to-date, market total **149 341**.

**How to read the output:**
* `month_ok` and `ytd_ok` must both be **True**: the brand rows add up exactly to CCFA's Total row.
* The BYD row must show 5 419 and 28 086.

In [ ]:
test_file = f"{RAW}/Immatriculations-VPN-Octobre-2026.pdf"
df_test, check_test = extract_brand_table(test_file, file_to_source.get(os.path.basename(test_file)))

print(check_test)
df_test[df_test.brand_label.isin(["BYD", "TESLA", "M.G.", "Total"])]

## Step 6 · Run the extraction on all files

**What it does:** loops over every PDF, extracts its brand table, and collects one check row per file.
**Why `try/except`:** if one file has an unexpected layout, the loop records the error and continues, so you see *all* problems at once instead of stopping at the first.
**What you should see:** `Files extracted: 34 | errors: 0`. If there are errors, send me the error table.

In [ ]:
frames, checks, errors = [], [], []
for path in pdfs:
    name = os.path.basename(path)
    try:
        df, chk = extract_brand_table(path, file_to_source.get(name))
        frames.append(df); checks.append(chk)
    except Exception as e:
        errors.append({"source_file": name, "error": str(e)})

brand_month_raw = pd.concat(frames, ignore_index=True)
checks_df = pd.DataFrame(checks).sort_values("period").reset_index(drop=True)

print(f"Files extracted: {len(frames)} | errors: {len(errors)}")
pd.DataFrame(errors)

## Step 7 · Validation 1: does each file add up?

**Rule:** inside each file, the sum of brand rows must equal CCFA's Total row, for the month **and** year-to-date.
**How to read it:** the table below lists **only failing files**. Empty = every file reconciles exactly.
**If a file fails:** the parser misread a row, *or* CCFA's own table doesn't add up. We investigate before going further and never "fix" numbers silently.

In [ ]:
print("Files checked:", len(checks_df))
print("All month totals reconcile:", checks_df.month_ok.all())
print("All YTD totals reconcile  :", checks_df.ytd_ok.all())
checks_df[~(checks_df.month_ok & checks_df.ytd_ok)]

## Step 8 · Validation 2: is every month present exactly once?

**Rule:** we expect 34 months, January 2024 to October 2026, with no gaps and no duplicates.
**Why:** a missing month makes growth rates wrong; a duplicate month double-counts registrations.
**How to read it:** both lists should be empty.

In [ ]:
expected = pd.period_range("2024-01", "2026-10", freq="M").strftime("%Y-%m")
found = checks_df["period"]
print("Missing months   :", sorted(set(expected) - set(found)))
print("Duplicate months :", sorted(found[found.duplicated()].unique()))
print("Unparsed periods :", checks_df[checks_df.period.isna()].source_file.tolist())

## Step 9 · Validation 3: does year-to-date carry over between months?

**Rule:** for each brand, `YTD previous month + this month = YTD this month` (in January, YTD = the month itself).
**Why it matters:** CCFA sometimes **revises earlier months** after publication. In Phase 1 we found Audi and the market total revised by 1–2 cars between August and September 2026. This check finds every such case automatically.

**How to read the output:**
* `gap` = published YTD − expected YTD. A small gap (±1–5) = a normal data revision.
* `AUTRES` gaps can be large: CCFA moves brands in and out of "other brands" and restates history. That is expected and documented.
* A large gap on a **named brand** needs investigation (possible extraction error or a brand merge such as ABARTH + FIAT).

We **log** revisions; we never overwrite the published figures.

In [ ]:
bm = brand_month_raw[~brand_month_raw.is_total_row].copy()
tot = brand_month_raw[brand_month_raw.is_total_row].copy()
tot["brand_label"] = "TOTAL MARKET"
panel = pd.concat([bm, tot]).sort_values(["brand_label", "period"])

panel["year"] = panel["period"].str[:4]
panel["prev_period"] = (pd.PeriodIndex(panel["period"], freq="M") - 1).strftime("%Y-%m")
prev = panel[["brand_label", "period", "registrations_ytd"]].rename(
    columns={"period": "prev_period", "registrations_ytd": "prev_ytd"})
panel = panel.merge(prev, on=["brand_label", "prev_period"], how="left")

is_jan = panel["period"].str.endswith("-01")
panel["expected_ytd"] = panel["prev_ytd"] + panel["registrations_month"]
panel.loc[is_jan, "expected_ytd"] = panel.loc[is_jan, "registrations_month"]
panel["gap"] = panel["registrations_ytd"] - panel["expected_ytd"]

revisions = panel[panel["gap"].notna() & (panel["gap"] != 0)][
    ["brand_label", "period", "prev_ytd", "registrations_month", "registrations_ytd", "expected_ytd", "gap", "source_id"]]
print("Brand-months checked:", panel["gap"].notna().sum(), "| with a revision gap:", len(revisions))
revisions.sort_values("gap", key=abs, ascending=False).head(25)

## Step 10 · First look at BYD (a sanity check, not analysis yet)

**What it does:** shows BYD's monthly registrations next to the total market, with BYD's share.
**Why:** a quick visual sense-check. Do the numbers look plausible and match Phase 1 (e.g. 5 419 in October 2026)?
**Data label:** `byd_share_pct` is **derived** (BYD ÷ total, both from the same CCFA table, so the scope matches).
Months where BYD was not listed separately would show as missing. That doesn't mean zero.

In [ ]:
byd = bm[bm.brand_label == "BYD"][["period", "registrations_month", "registrations_ytd", "source_id"]]
market = tot[["period", "registrations_month"]].rename(columns={"registrations_month": "market_total"})
byd = pd.DataFrame({"period": expected}).merge(byd, on="period", how="left").merge(market, on="period", how="left")
byd["byd_share_pct"] = (byd["registrations_month"] / byd["market_total"] * 100).round(2)
byd

## Step 11 · Save outputs

**What it does:** writes the three output files to `data/processed/`.
**Then:** download them (cell below) and upload them to the same folder on GitHub, so the repo holds the code **and** its results.

In [ ]:
cols = ["source_id", "source_file", "period", "month_label", "brand_label", "is_total_row",
        "registrations_month", "var_month_pct_reported", "registrations_ytd", "var_ytd_pct_reported"]
brand_month_raw = brand_month_raw.sort_values(["period", "is_total_row", "brand_label"])[cols]
brand_month_raw.to_csv(f"{PROCESSED}/brand_month_raw.csv", index=False)
checks_df.to_csv(f"{PROCESSED}/extraction_checks.csv", index=False)
revisions.to_csv(f"{PROCESSED}/ytd_revisions_log.csv", index=False)

print("Saved:", len(brand_month_raw), "rows to brand_month_raw.csv")
from google.colab import files
for f in ["brand_month_raw.csv", "extraction_checks.csv", "ytd_revisions_log.csv"]:
    files.download(f"{PROCESSED}/{f}")

## What this notebook achieved, and what's next

**Done:** 34 monthly PDFs → one traceable table, with three automated quality checks (totals, completeness, YTD continuity).

**Interview talking point:** *"Before building KPIs I reconciled every extracted file to the publisher's own totals, checked month coverage, and logged year-to-date revisions instead of overwriting them. That's how I found CCFA restates earlier months and reallocates brands out of 'Others'."*

**Next, notebook 02 (cleaning):** standardise brand names over time (`ABARTH` + `FIAT` → `ABARTH/FIAT`, `M.G.` → `MG`), flag new-brand months, and prepare the tables for SQL.